# Swin detection-only fine-tuning at 512×768
Warm-start from the verified Swin detector checkpoint, or a DINO bundle containing that detector. Train detection only for 50 epochs. Random frame batches (not P-K) include all annotated people. No ReID training or crop encoding. Preserve the initial frame split seed; change no annotation labels.

Validation selects mean(AP50, AP75), each 101-point interpolated AP at a 0.001 candidate floor. This is not COCO AP50:95. Precision/recall use confidence 0.5. Final test is detection-only. For future person-search evaluation, load the saved scene resolution too; the older retrieval default of 256×384 must not be used silently.


In [ ]:
from kaggle_secrets import UserSecretsClient
import base64
import os
import subprocess
import sys
from pathlib import Path

USER = "SaadQAbbasi"
REPO = "ML4CV-Assignment"
repo_path = f"/kaggle/working/{REPO}"
repo_dir = Path(repo_path)
remote = f"https://github.com/{USER}/{REPO}.git"

# Pass credentials through the child environment, not the URL or notebook output.
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({
    "GIT_TERMINAL_PROMPT": "0",
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}",
})
try:
    if not repo_dir.exists():
        command = ["git", "clone", remote, str(repo_dir)]
    elif (repo_dir / ".git").is_dir():
        command = ["git", "-C", str(repo_dir), "pull", "--ff-only", remote]
    else:
        raise RuntimeError(f"{repo_dir} exists but is not a Git checkout.")
    result = subprocess.run(command, env=git_env, capture_output=True, text=True)
    if result.returncode:
        detail = (result.stderr + result.stdout).replace(token, "[REDACTED]").replace(auth, "[REDACTED]")
        raise RuntimeError(
            "Repository setup failed. Check Kaggle Internet access, GITHUB_TOKEN "
            "secret access for this notebook, and repository permissions.\n" + detail
        )
finally:
    del token, auth, git_env

src_dir = repo_dir / "src"
required = ("model.py", "prw_dataset.py", "training.py", "prw_evaluation.py", "dino_reid.py", "dinov3_reid.py", "offline_mining.py")
missing = [name for name in required if not (src_dir / name).is_file()]
if missing:
    raise RuntimeError(f"Repository at {repo_dir} is missing source files: {missing}")
src_path = str(src_dir.resolve())
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)
import importlib
importlib.invalidate_caches()
print(f"Project imports configured from {src_path}")


In [ ]:


from kaggle_secrets import UserSecretsClient
import wandb

secrets = UserSecretsClient()
wandb_api_key = secrets.get_secret("WAND_B_API_KEY")

wandb.login(key=wandb_api_key)




PRW_ROOT = "/kaggle/input/datasets/edoardomerli/prw-person-re-identification-in-the-wild"

data_root = PRW_ROOT



In [ ]:
import random
import json
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from model import PersonSearchModel
from prw_dataset import PRWPersonSearchDataset, PRWSubset, prw_collate_fn
from training import make_grad_scaler, start_run
from detection_training import detection_epoch, evaluate_detector

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SOURCE_CHECKPOINT=Path('/kaggle/input/YOUR-SWIN-OR-DINO-CHECKPOINT/best.pt')
IMAGE_SIZE=(512,768)
BATCH_SIZE=4
NUM_EPOCHS=50
NUM_WORKERS=2
BACKBONE_LR=1e-5
HEAD_LR=1e-4
UPLOAD_EVERY=10
state=torch.load(SOURCE_CHECKPOINT,map_location='cpu',weights_only=False)
config=dict(state.get('detector_config',state.get('model_config',{})))
if config.get('backbone_name')!='swin_t':
    raise ValueError('Expected a verified Swin-T detector checkpoint')
SEED=int(state['run_config']['seed'])
VAL_FRACTION=float(state['run_config'].get('val_fraction',.10))
random.seed(SEED);torch.manual_seed(SEED)
config.update(pretrained=False,detection_only=True)
model=PersonSearchModel(**config).to(device)
model.load_state_dict(state.get('detector_state_dict',state.get('model_state_dict')),strict=True)
del state
full=PRWPersonSearchDataset(PRW_ROOT,split='train',image_size=IMAGE_SIZE,include_query_crops=False)
indices=torch.randperm(len(full),generator=torch.Generator().manual_seed(SEED)).tolist()
cut=round(len(indices)*VAL_FRACTION)
train_data=PRWSubset(full,indices[cut:]);val_data=PRWSubset(full,indices[:cut])
def loader(data,shuffle=False):
    return DataLoader(data,batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=NUM_WORKERS,
                      collate_fn=prw_collate_fn,pin_memory=device.type=='cuda')
train_loader=loader(train_data,True);val_loader=loader(val_data)
print('Train/validation frames:',len(train_data),len(val_data),'seed:',SEED)
run_config=dict(experiment='swin_detection_only_512x768',seed=SEED,val_fraction=VAL_FRACTION,
                image_size=IMAGE_SIZE,batch_size=BATCH_SIZE,epochs=NUM_EPOCHS,
                source_checkpoint=str(SOURCE_CHECKPOINT),model_config=config,
                backbone_lr=BACKBONE_LR,head_lr=HEAD_LR,
                checkpoint_selection='mean_AP50_AP75')
run=start_run(run_config,name='swin-detector-only-512x768',mode='online')
# Baseline at the new resolution before any detector updates.
baseline=evaluate_detector(model,val_loader,device)
run.log({'epoch':0,**{f'val/{k}':v for k,v in baseline.items()}})
print('Initial validation:',baseline)


In [ ]:
optimizer=torch.optim.AdamW([
    {'params':[p for p in model.backbone.parameters() if p.requires_grad],'lr':BACKBONE_LR},
    {'params':[p for name,p in model.named_parameters()
               if p.requires_grad and not name.startswith('backbone.')],'lr':HEAD_LR},
],weight_decay=1e-4)
scaler=make_grad_scaler(device)
checkpoint_dir=Path(repo_path)/'checkpoints_swin_detection_512x768'
checkpoint_dir.mkdir(parents=True,exist_ok=True)
best=baseline['mean_AP50_AP75']
def save_checkpoint(epoch,metrics,path):
    torch.save(dict(epoch=epoch,model_config=config,model_state_dict=model.state_dict(),
                    optimizer_state_dict=optimizer.state_dict(),scaler_state_dict=scaler.state_dict(),
                    run_config=run_config,detector_image_size=IMAGE_SIZE,
                    best_detection_ap=best,validation_metrics=metrics),path)
# Preserve warm-start weights if fine-tuning never improves validation.
save_checkpoint(0,baseline,checkpoint_dir/'best.pt')
for epoch in range(1,NUM_EPOCHS+1):
    if device.type=='cuda':torch.cuda.reset_peak_memory_stats(device)
    losses=detection_epoch(model,train_loader,optimizer,scaler,device)
    payload={'epoch':epoch,**{f'train/{k}':v for k,v in losses.items()}}
    if device.type=='cuda':
        payload['train/peak_allocated_gib']=torch.cuda.max_memory_allocated(device)/2**30
        payload['train/peak_reserved_gib']=torch.cuda.max_memory_reserved(device)/2**30
    metrics=evaluate_detector(model,val_loader,device)
    payload.update({f'val/{k}':v for k,v in metrics.items()})
    run.log(payload)
    if metrics['mean_AP50_AP75']>best:
        best=metrics['mean_AP50_AP75'];save_checkpoint(epoch,metrics,checkpoint_dir/'best.pt')
    save_checkpoint(epoch,metrics,checkpoint_dir/'latest.pt')
    print(f'Epoch {epoch}: {metrics}; best={best:.4f}',flush=True)
    if epoch%UPLOAD_EVERY==0 or epoch==NUM_EPOCHS:
        for file,collection in [('best.pt','swin-detector-512x768-best'),('latest.pt','swin-detector-512x768-latest')]:
            artifact=wandb.Artifact(collection,type='model',metadata={'upload_epoch':epoch})
            artifact.add_file(str(checkpoint_dir/file));run.log_artifact(artifact)


## Final detection test
Load the validation-selected detector and evaluate official test frames once. This does not evaluate ReID. The checkpoint contains the detector input resolution for later controlled Swin/DINO retrieval comparisons.


In [ ]:
best_state=torch.load(checkpoint_dir/'best.pt',map_location='cpu',weights_only=False)
model.load_state_dict(best_state['model_state_dict'],strict=True)
selected_epoch=best_state['epoch']
del best_state
test_data=PRWPersonSearchDataset(PRW_ROOT,split='test',image_size=IMAGE_SIZE,include_query_crops=False)
test_metrics=evaluate_detector(model,loader(test_data),device)
output=checkpoint_dir/'test_detection.json'
output.write_text(json.dumps(dict(checkpoint_epoch=selected_epoch,metrics=test_metrics),indent=2))
try:
    payload={f'test_detection/{k}':v for k,v in test_metrics.items()}
    payload['test_detection/checkpoint_epoch']=selected_epoch
    run.summary.update(payload);run.log(payload)
    table=wandb.Table(columns=['metric','value'],data=[[k,v] for k,v in test_metrics.items()])
    run.log({'test_detection/results':table,
             'test_detection/chart':wandb.plot.bar(table,'metric','value',title='Detection test metrics')})
    print('Selected epoch:',selected_epoch,'FINAL DETECTION TEST:',test_metrics)
finally:
    run.finish()
